# Environment set up

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import re
import os
import json
from datasets import Dataset
import numpy as np

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# !pip install transformers==4.56.2
!pip install --upgrade transformers huggingface_hub datasets
!pip install --no-deps trl==0.22.2

In [ ]:
import torch
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
from transformers import TrainingArguments, Trainer
from trl import SFTTrainer
from unsloth import tokenizer_utils
from peft import PeftModel
import torch.nn as nn
import torch.nn.functional as F

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# GLOBAL VARIABLES
CATEGORIES_PATH = "/content/drive/MyDrive/Projects/Assignment_02/data_raw/categories.json"
TRAIN_DATASET_PATH = "/content/drive/MyDrive/Projects/Assignment_02/data_raw/train.csv"
TEST_DATASET_PATH = "/content/drive/MyDrive/Projects/Assignment_02/data_raw/test.csv"

# Data Pre-processing

In [ ]:
def normalize_text(text):
    if not isinstance(text, str):
        return str(text)

    text = re.sub(r"[‘'´`’]", "'", text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip().lower()
    return text


# Training session

## Load the baseline model

In [ ]:
def do_nothing(*args, **kwargs):
    pass
tokenizer_utils.fix_untrained_tokens = do_nothing
with open(CATEGORIES_PATH, "r") as f:
    categories = json.load(f)
NUM_CLASSES = len(categories) # 77 classes

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
number_token_ids = []

for i in range(0, NUM_CLASSES):
    number_token_ids.append(tokenizer.encode(str(i), add_special_tokens=False)[0])

old_shape = model.lm_head.weight.shape
old_size = old_shape[0]

par = torch.nn.Parameter(model.lm_head.weight[number_token_ids, :].clone().detach())
model.lm_head.weight = par

reverse_map = {value: idx for idx, value in enumerate(number_token_ids)}
reverse_map

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.6.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

[transformers] Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Đã thu nhỏ lm_head. Bộ từ điển có 77 keys.


{15: 0,
 16: 1,
 17: 2,
 18: 3,
 19: 4,
 20: 5,
 21: 6,
 22: 7,
 23: 8,
 24: 9,
 605: 10,
 806: 11,
 717: 12,
 1032: 13,
 975: 14,
 868: 15,
 845: 16,
 1114: 17,
 972: 18,
 777: 19,
 508: 20,
 1691: 21,
 1313: 22,
 1419: 23,
 1187: 24,
 914: 25,
 1627: 26,
 1544: 27,
 1591: 28,
 1682: 29,
 966: 30,
 2148: 31,
 843: 32,
 1644: 33,
 1958: 34,
 1758: 35,
 1927: 36,
 1806: 37,
 1987: 38,
 2137: 39,
 1272: 40,
 3174: 41,
 2983: 42,
 3391: 43,
 2096: 44,
 1774: 45,
 2790: 46,
 2618: 47,
 2166: 48,
 2491: 49,
 1135: 50,
 3971: 51,
 4103: 52,
 4331: 53,
 4370: 54,
 2131: 55,
 3487: 56,
 3226: 57,
 2970: 58,
 2946: 59,
 1399: 60,
 5547: 61,
 5538: 62,
 5495: 63,
 1227: 64,
 2397: 65,
 2287: 66,
 3080: 67,
 2614: 68,
 3076: 69,
 2031: 70,
 6028: 71,
 5332: 72,
 5958: 73,
 5728: 74,
 2075: 75,
 4767: 76}

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,

    target_modules = ["lm_head", "q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
FastLanguageModel.for_training(model)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:919: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(
[transformers] Unsloth 2026.4.8 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

## Data preparation

In [ ]:
os.makedirs("configs", exist_ok=True)

with open(CATEGORIES_PATH, "r") as f:
    categories = json.load(f)

label2id = {label: idx for idx, label in enumerate(categories)}
id2label = {idx: label for label, idx in label2id.items()}

print("Label to ID:", label2id)
print("ID to Label:", id2label)

train_df = pd.read_csv(TRAIN_DATASET_PATH)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df['text'] = train_df['text'].apply(normalize_text)
train_df['label'] = train_df['category'].map(label2id)


print(train_df[['text', 'category', 'label']].head(5))

mapping_data = {
    "label2id": label2id,
    "id2label": id2label
}
with open("configs/label_mapping.json", "w") as f:
    json.dump(mapping_data, f, indent=4)
print("\nĐã lưu 'configs/label_mapping.json'")

In [ ]:
from datasets import Dataset
import json

with open(CATEGORIES_PATH, "r") as f:
    categories = json.load(f)

numbered_categories = [f"class {i}: {label}" for i, label in enumerate(categories)]
categories_str = "\n".join(numbered_categories)

banking_prompt = """Here is a banking query:
{input_text}

Classify this query into one of the following intents:

{categories_list}

SOLUTION
The correct answer is: class """

def manual_tokenize_and_format(examples):
    inputs  = examples["text"]
    outputs = examples["category"]

    input_ids_list = []
    attention_mask_list = []

    for input_text, output_text in zip(inputs, outputs):
        safe_input = str(input_text)[:1500]
        class_id_number = label2id[output_text]

        target_token_id = number_token_ids[class_id_number]

        prompt_str = banking_prompt.format(
            categories_list = categories_str,
            input_text = safe_input
        )

        tokenized = tokenizer(prompt_str, add_special_tokens=True)
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]

        input_ids.append(target_token_id)
        attention_mask.append(1)

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list
    }

hf_train_dataset = Dataset.from_pandas(train_df[['text', 'category']])
tokenized_dataset = hf_train_dataset.map(
    manual_tokenize_and_format,
    batched = True,
    remove_columns = hf_train_dataset.column_names,
    load_from_cache_file = False
)

Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

## Train the model

In [ ]:
from typing import Any, Dict, List, Union
from transformers import DataCollatorForLanguageModeling
from trl import SFTConfig, SFTTrainer

class DataCollatorForLastTokenLM(DataCollatorForLanguageModeling):
    def __init__(self, *args, mlm: bool = False, ignore_index: int = -100, **kwargs):
        super().__init__(*args, mlm=mlm, **kwargs)
        self.ignore_index = ignore_index

    def torch_call(self, examples: List[Union[List[int], Any, Dict[str, Any]]]) -> Dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["labels"][i]
            non_masked = (labels != self.ignore_index).nonzero(as_tuple=True)[0]
            if len(non_masked) == 0:
                continue
            last_token_idx = non_masked[-1].item()
            original_label = labels[last_token_idx].item()

            batch["labels"][i, :last_token_idx] = self.ignore_index

            if original_label in reverse_map:
                batch["labels"][i, last_token_idx] = reverse_map[original_label]
            else:
                batch["labels"][i, last_token_idx] = self.ignore_index

        return batch

collator = DataCollatorForLastTokenLM(tokenizer=tokenizer)


In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/Projects/Assignment_02/outputs"
training_args = TrainingArguments(
    output_dir=OUTPUT_PATH,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    num_train_epochs=1,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="cosine",
    seed=3407,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=collator,
)

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(tokenized_dataset, batch_size=2, collate_fn=collator)
batch = next(iter(loader))

labels = batch["labels"][0]
non_masked = labels[labels != -100]
print("Non-masked labels (should all be 0-76):", non_masked)
assert all(0 <= x <= 76 for x in non_masked.tolist()), "Labels out of range!"
print("Safe to train.")

In [ ]:
trainer_stats = trainer.train()

[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,003 | Num Epochs = 1 | Total steps = 626
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 13,357,056 of 1,249,329,152 (1.07% trained)


Step,Training Loss
10,2.167890
20,1.629364
30,1.327851
40,1.220734
50,1.312179
60,1.737648
70,1.559657
80,1.758743
90,1.203732
100,1.089174


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Projects/Assignment_02/outputs/checkpoint-100/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Projects/Assignment_02/outputs/checkpoint-200/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` 

In [ ]:
import shutil
from google.colab import files

model.save_pretrained_merged("banking-intent-llama-3.2", tokenizer, save_method="merged_16bit")

import json
with open("banking-intent-llama-3.2/label_mapping.json", "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=4)

# Zip and download
shutil.make_archive("model_finetuned", "zip", "banking-intent-llama-3.2")
files.download("model_finetuned.zip")
print("Done. Check your browser downloads.")

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

[transformers] Unsloth: Restored added_tokens_decoder metadata in banking-intent-llama-3.2/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

...

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:52<00:00, 52.48s/it]


Unsloth: Merge process complete. Saved to `/content/banking-intent-llama-3.2`


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done. Check your browser downloads.


## Train from a checkpoint

In [ ]:
CHECKPOINT_PATH = ""
trainer_stats = trainer.train(resume_from_checkpoint = CHECKPOINT_PATH)

[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,003 | Num Epochs = 3 | Total steps = 3,753
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
3010,1.550055
3020,1.580608
3030,1.480191
3040,1.489313
3050,1.438444
3060,1.550516
3070,1.382906
3080,1.476304
3090,1.457813
3100,1.546132


[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Projects/Assignment_02/outputs/checkpoint-3300/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Projects/Assignment_02/outputs/checkpoint-3600/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Projects/Assignment_02/outputs/checkpoint-3753/tokenizer_config.json.


# Inference

In [ ]:
import json, re, torch, torch.nn.functional as F
from unsloth import FastLanguageModel, tokenizer_utils

HF_REPO     = "juzharii/banking-intent-llama-3.2"
NUM_CLASSES = 77

def do_nothing(*args, **kwargs): pass
tokenizer_utils.fix_untrained_tokens = do_nothing

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_REPO,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
if model.lm_head.weight.shape[0] != NUM_CLASSES:
    number_token_ids = [
        tokenizer.encode(str(i), add_special_tokens=False)[0]
        for i in range(NUM_CLASSES)
    ]
    hidden_dim    = model.lm_head.in_features
    pruned_weight = model.lm_head.weight.data[number_token_ids, :].clone().detach()
    new_lm_head   = torch.nn.Linear(hidden_dim, NUM_CLASSES, bias=False, device=model.device)
    new_lm_head.weight = torch.nn.Parameter(pruned_weight.to(new_lm_head.weight.dtype))
    model.lm_head = new_lm_head

FastLanguageModel.for_inference(model)
print(f"lm_head shape: {model.lm_head.weight.shape}")  # should print [77, hidden_dim]

# Load label mapping
from huggingface_hub import hf_hub_download
label_path = hf_hub_download(repo_id=HF_REPO, filename="label_mapping.json")
with open(label_path) as f:
    mapping = json.load(f)
id2label = {int(k): v for k, v in mapping["id2label"].items()}

# Predict
categories_str = "\n".join([f"class {i}: {id2label[i]}" for i in range(NUM_CLASSES)])

banking_prompt = """Here is a banking query:
{input_text}

Classify this query into one of the following intents:

{categories_list}

SOLUTION
The correct answer is: class """

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r"[''´`']", "'", text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

def predict_intent(user_query: str):
    prompt = banking_prompt.format(
        input_text=normalize_text(user_query)[:1500],
        categories_list=categories_str,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model(**inputs)

    last_token_logits = outputs.logits[0, -1, :]
    pred_id    = torch.argmax(last_token_logits).item()
    confidence = F.softmax(last_token_logits, dim=-1)[pred_id].item()
    return id2label[pred_id], confidence

# Smoke test
for q in ["I lost my card", "What is the exchange rate for USD to EUR?"]:
    intent, conf = predict_intent(q)
    print(f"Q: {q}\n→ {intent} ({conf:.3f})\n")

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.6.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Unsloth: Will load juzharii/banking-intent-llama-3.2 as a legacy tokenizer.


lm_head shape: torch.Size([77, 2048])


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Q: I lost my card
→ lost_or_stolen_card (0.878)

Q: What is the exchange rate for USD to EUR?
→ exchange_rate (0.988)



# Evaluation

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

test_df = pd.read_csv(TEST_DATASET_PATH)
test_df['text'] = test_df['text'].apply(normalize_text)

y_true, y_pred = [], []

for text, label in tqdm(zip(test_df['text'], test_df['category']),
                         total=len(test_df), desc="Evaluating"):
    intent, _ = predict_intent(text)
    y_true.append(label)
    y_pred.append(intent)

acc        = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f"Accuracy : {acc * 100:.2f}%")
print(f"Precision: {p  * 100:.2f}%")
print(f"Recall   : {r  * 100:.2f}%")
print(f"F1-Score : {f1 * 100:.2f}%")
print()
print(classification_report(y_true, y_pred, zero_division=0))